In [1]:
import torch
import stable_worldmodel as swm
from sigreg import SIGReg 
from torch.distributions.normal import Normal
from torch.linalg import vector_norm
from model import WorldModel
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F
from stable_worldmodel.wm.utils import save_pretrained
from omegaconf import OmegaConf
import trackio

In [8]:
device = 'mps'
num_epochs = 10
batch_size = 54
lr = 2e-4
lmbd = 1

In [10]:
dataset = swm.data.load_dataset(
    'tworooms.lance',
    num_steps=5,
    frameskip=1,
    keys_to_load=['pixels', 'action', 'state'],
)
test_dataset = swm.data.load_dataset(
    'tworooms-test.lance',
    num_steps=5,
    frameskip=1,
    keys_to_load=['pixels', 'action', 'state'],
)

FileNotFoundError: Cannot resolve 'tworooms-test.lance': not a local path or HF repo id.

In [11]:
action_dim = 2

In [5]:
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    drop_last=True,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    drop_last=True,
    pin_memory=True
)

NameError: name 'dataset' is not defined

In [7]:
len(loader)/54

88.29629629629629

In [12]:
model = WorldModel(
    hidden_dim=192, 
    patch_size=14,
    channels=3, 
    enc_nb_heads=3, 
    enc_nb_layers=12, 
    height=224, 
    width=224, 
    pred_nb_layers=6, 
    pred_p_dropout=0.1, 
    pred_nb_heads=16, 
    action_dim=action_dim
).to(device)

In [15]:
model.load_state_dict(torch.load('./tworoom-fov-training-0.pt', map_location=torch.device('mps')))

<All keys matched successfully>

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

In [9]:
trackio.init(
    project="tworoom",
    name="training-6",
    config={"epochs": num_epochs, "learning_rate": lr, "batch_size": batch_size, "lambda": lmbd},
    server_url="http://127.0.0.1:7862?write_token=3qxFCeHCqCKLF6FUAfE5D-4OxPJjzoHP7TmHAVf10oE"
)

* Trackio project initialized: tworoom
* Trackio metrics will be sent to self-hosted server: http://127.0.0.1:7862


* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: training-6


In [10]:
for epoch in range(num_epochs):
    for i, batch in enumerate(loader):
        optimizer.zero_grad()
        frames = batch['pixels'].to(device).type(torch.float32)
        actions = torch.nan_to_num(batch['action'], 0.0).to(device).type(torch.float32)
        pred, embeds = model(frames, actions)
        t_sigregloss = SIGReg(embeds.reshape(-1, embeds.shape[-1]))
        t_mseloss = F.mse_loss(pred[:,:-1,:], embeds[:,1:,:])
        t_loss = t_mseloss + lmbd*t_sigregloss
        t_loss.backward()
        optimizer.step()
        with torch.no_grad():
            model.eval()
            val_batch = next(iter(test_loader))
            frames = val_batch['pixels'].to(device).type(torch.float32)
            actions = torch.nan_to_num(val_batch['action'], 0.0).to(device).type(torch.float32)
            pred, embeds = model(frames, actions)
            v_sigregloss = SIGReg(embeds.reshape(-1, embeds.shape[-1]))
            v_mseloss = F.mse_loss(pred[:,:-1,:], embeds[:,1:,:])
            val_sigregloss = v_sigregloss
            val_mseloss = v_mseloss
            val_loss = v_mseloss + lmbd*v_sigregloss
            model.train()
        trackio.log({
            "epoch": epoch,
            "batch": i,
            "train_mse_loss": t_mseloss.item(),
            "train_sigreg_loss": t_sigregloss.item(),
            "train_loss": t_loss.item(),
            "val_mse_loss": val_mseloss.item(),
            "val_sigreg_loss": val_sigregloss.item(),
            "val_loss": val_loss.item()
        })

    scheduler.step()
trackio.finish()

ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/workspace/leworldmodel/.venv/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/workspace/leworldmodel/.venv/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/leworldmodel/.venv/lib/python3.11/site-packages/stable_worldmodel/data/formats/lance.py", line 492, in __getitems__
    self._ensure_open()
  File "/workspace/leworldmodel/.venv/lib/python3.11/site-packages/stable_worldmodel/data/formats/lance.py", line 319, in _ensure_open
    table = self._connect_table()
            ^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/leworldmodel/.venv/lib/python3.11/site-pac

In [ ]:
torch.save(model.state_dict(), f'tworoom-6-{num_epochs}.pt')

In [16]:
model_config = OmegaConf.create({
    '_target_': 'model.WorldModel',
    'hidden_dim': 192,
    'action_dim': 2,
    'patch_size': 14,
    'channels':3, 
    'enc_nb_heads':3, 
    'enc_nb_layers':12, 
    'height':224, 
    'width':224, 
    'pred_nb_layers':6, 
    'pred_p_dropout':0.1, 
    'pred_nb_heads':16
})

save_pretrained(
    model=model,
    run_name='tworoom-fov',
    config=model_config,
    filename='training-0-alt.pt',
)


2026-06-22 22:30:03.805 | INFO     | stable_worldmodel.wm.utils:save_pretrained:42 - 📦📦📦 Model saved to /Users/matheoledevehat/.stable_worldmodel/checkpoints/tworoom-fov/training-0-alt.pt 📦📦📦
